# Làm sạch có nhật ký: vì sao doanh thu của bạn lệch số của kế toán?

Bộ `online-retail-ii`, sheet **`Year 2009-2010`** (525 461 dòng, 01/12/2009 tới 09/12/2010; lý thuyết module dùng
sheet năm sau). Mỗi dòng là một mặt hàng trong một hoá đơn: `Invoice`, `StockCode`, `Description`, `Quantity`,
`InvoiceDate`, `Price` (GBP/đơn vị), `Customer ID`, `Country`.

Kế toán trưởng hỏi: **"Báo cáo doanh thu hàng hoá của team data ra một số, sổ của tôi ra số khác. Các bạn đã bỏ
những dòng nào, bao nhiêu tiền?"** Câu trả lời tốt không phải là "em đã làm sạch dữ liệu", mà là một **bảng nhật
ký**: mỗi bước loại dòng có lý do, số dòng bị loại, số dòng còn lại, số tiền bị loại; cộng lại khớp từ số thô tới
số cuối.

Việc của bạn: chuẩn hoá kiểu và chuỗi, viết hàm làm sạch có nhật ký, hồ sơ dữ liệu thiếu, rồi **làm phần trắc
nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/basic/01-retail-cleaning-log -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2009-2010
# xlsx đọc mất 30-60 giây: lần đầu lưu bản parquet vào thư mục tạm, các lần sau đọc trong vài giây
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_sheet(sheet):
    src = ensure("online-retail-ii") / "online_retail_II.xlsx"
    st = src.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"online_retail_II-{sheet.replace(' ', '_')}-{st.st_size}-{st.st_mtime_ns}.parquet"
    if not cache.exists():
        df = pd.read_excel(src, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
        cache.parent.mkdir(parents=True, exist_ok=True)
        tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
        df.to_parquet(tmp, index=False)
        os.replace(tmp, cache)  # ghi atomic: không để lại tệp dở dang
    return pd.read_parquet(cache)


tx = read_sheet("Year 2009-2010")
print(tx.shape)
print(tx.dtypes)
tx.head()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Mã không phải hàng hoá: phí ship, phí sàn, điều chỉnh, test. Đã viết hoa (so sau khi chuẩn hoá StockCode).
FEE_CODES = {
    "POST", "DOT", "M", "C2", "D", "S", "B", "BANK CHARGES", "AMAZONFEE", "CRUK", "PADS",
    "ADJUST", "ADJUST2", "TEST001", "TEST002",
}
# Thứ tự các bước: mỗi dòng bị loại được ghi cho lý do ĐẦU TIÊN nó vi phạm.
REASONS = ["cancelled", "non_positive_quantity", "non_positive_price", "fee_code", "exact_duplicate"]

## 1. Chuẩn hoá kiểu và chuỗi (transformation)

Excel đọc cùng một cột thành nhiều kiểu: mã hoá đơn thuần số thành `int`, mã có chữ (`C489449`, `A506401`) thành
`str`; `Customer ID` thành `float` vì có ô trống (`13085.0`). Chuỗi có khoảng trắng thừa ở đầu, cuối, giữa; mã hàng
có bản chữ thường (`85123a`) và chữ hoa (`85123A`) cho cùng một sản phẩm, mã phí có `m` lẫn `M`.

`normalize(tx) -> DataFrame` trả bản sao, cùng index, với:

- `Invoice`: `str`, bỏ khoảng trắng hai đầu.
- `StockCode`: `str`, bỏ khoảng trắng hai đầu, **viết hoa**.
- `Description`: bỏ khoảng trắng hai đầu, gộp nhiều khoảng trắng liền nhau thành một; ô trống giữ là thiếu.
- `Customer ID`: kiểu `Int64` (số nguyên cho phép thiếu, `<NA>`).
- `Country`: `str`, bỏ khoảng trắng hai đầu.

In [ ]:
def normalize(tx: pd.DataFrame) -> pd.DataFrame:
    """Bản sao của tx với kiểu và chuỗi đã chuẩn hoá (xem mô tả ở trên)."""
    # TODO: out = tx.copy()
    # TODO: Invoice, StockCode (viết hoa), Description (strip + gộp khoảng trắng), Customer ID (Int64), Country
    raise NotImplementedError

In [ ]:
# explore: chuẩn hoá thay đổi bao nhiêu thứ?
norm = normalize(tx)
print("mã hàng khác nhau:", tx["StockCode"].astype(str).nunique(), "→", norm["StockCode"].nunique())
print("mô tả khác nhau:", tx["Description"].nunique(), "→", norm["Description"].nunique())
lower = tx["StockCode"].astype(str).str.strip() != norm["StockCode"]
print(f"{lower.sum():,} dòng có mã hàng chữ thường, ví dụ:", tx.loc[lower, "StockCode"].value_counts().head(5).to_dict())

## 2. Làm sạch có nhật ký

`clean_with_log(tx) -> (clean, log)`:

1. Gọi `normalize(tx)` trước (mã `m` phải bị bắt như `M`; hai dòng chỉ khác khoảng trắng là trùng).
2. Lần lượt theo `REASONS`, mỗi bước bỏ các dòng **còn lại** vi phạm quy tắc:
   - `cancelled`: `Invoice` bắt đầu bằng `C` (hoá đơn huỷ);
   - `non_positive_quantity`: `Quantity <= 0`;
   - `non_positive_price`: `Price <= 0` (hàng tặng, điều chỉnh nợ xấu `A...` giá âm);
   - `fee_code`: `StockCode` thuộc `FEE_CODES` (phí, không phải hàng hoá);
   - `exact_duplicate`: dòng trùng **mọi cột** với một dòng đứng trước nó, giữ dòng đầu.
3. `clean`: các dòng còn lại (giữ index gốc, các cột như `normalize`).
4. `log`: một dòng mỗi lý do theo đúng thứ tự, cột `reason`, `rows_removed`, `rows_left`, `revenue_removed`
   (tổng `Quantity * Price` của các dòng bị loại ở bước đó; có thể âm).

Thiếu `Customer ID` **không** phải lý do loại: báo cáo doanh thu không cần biết khách là ai. Ba quy tắc đầu là
"dòng hợp lệ" theo định nghĩa dùng chung của track; hai quy tắc sau là bước làm sạch thêm cho **doanh thu hàng
hoá** của bài này.

In [ ]:
def clean_with_log(tx: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(clean, log): dòng còn lại sau 5 bước và nhật ký số dòng / số tiền bị loại theo lý do."""
    # TODO: df = normalize(tx); rows = []
    # TODO: với mỗi reason trong REASONS: tạo mặt nạ trên df hiện tại, ghi rows_removed, rows_left,
    #       revenue_removed (Quantity * Price của dòng bị loại), rồi df = df[~mask]
    # TODO: trả df, pd.DataFrame(rows, columns=["reason", "rows_removed", "rows_left", "revenue_removed"])
    raise NotImplementedError

In [ ]:
# explore: nhật ký và bảng đối soát từ số thô tới số cuối
clean, log = clean_with_log(tx)
print(log.to_string(index=False))
raw_revenue = (tx["Quantity"] * tx["Price"]).sum()
clean_revenue = (clean["Quantity"] * clean["Price"]).sum()
print(f"tổng Quantity * Price thô: {raw_revenue:,.2f}")
print(f"trừ phần bị loại:          {log['revenue_removed'].sum():,.2f}")
print(f"doanh thu hàng hoá:        {clean_revenue:,.2f}")
valid = log.loc[log.reason == "non_positive_price", "rows_left"].item()
print(f"dòng hợp lệ (định nghĩa dùng chung): {valid:,}; còn lại sau 5 bước: {len(clean):,}")

In [ ]:
# explore: số dòng bị loại theo lý do, bar ngang sắp theo thứ tự các bước
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(log["reason"], log["rows_removed"], color="#4c78a8")
ax.invert_yaxis()
for y, v in enumerate(log["rows_removed"]):
    ax.annotate(f"{v:,}", (v, y), xytext=(3, 0), textcoords="offset points", va="center", fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{len(tx) - len(clean):,} dòng bị loại ({(len(tx) - len(clean)) / len(tx):.1%}), theo lý do đầu tiên", loc="left")
plt.show()

In [ ]:
# explore: phí nào lớn nhất? (doanh thu dòng phí trong các dòng hợp lệ)
valid_rows = normalize(tx)
valid_rows = valid_rows[~valid_rows["Invoice"].str.startswith("C") & (valid_rows["Quantity"] > 0) & (valid_rows["Price"] > 0)]
fees = valid_rows[valid_rows["StockCode"].isin(FEE_CODES)]
(fees["Quantity"] * fees["Price"]).groupby(fees["StockCode"]).agg(["sum", "size"]).sort_values("sum", ascending=False).head(6)

## 3. Hồ sơ dữ liệu thiếu

`missing_report(df) -> DataFrame`: một dòng mỗi cột của `df`, cột `missing` (số ô thiếu, `int`) và
`missing_share` (tỉ lệ trên số dòng), sắp theo `missing` giảm dần; cột bằng nhau giữ thứ tự gốc.

In [ ]:
def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    """Số ô thiếu và tỉ lệ thiếu theo cột, sắp giảm dần."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thiếu ở đâu, trước và sau làm sạch?
print(missing_report(tx).head(3))
print(missing_report(clean).head(3))

## 4. Đọc nhật ký, rồi làm trắc nghiệm

Trước khi gửi cho kế toán trưởng, tự trả lời:

1. Bước nào loại nhiều **dòng** nhất, bước nào loại nhiều **tiền** nhất? Vì sao hai câu trả lời khác nhau?
2. Bước `fee_code` loại doanh thu dương: kế toán có tính phí ship vào doanh thu không? Đây là quyết định
   **định nghĩa**, không phải lỗi dữ liệu; ghi vào nhật ký để người đọc chọn.
3. Dòng trùng hoàn toàn có chắc là lỗi? Một khách quét cùng món hai lần trong một hoá đơn cũng tạo ra hai dòng
   giống hệt. Bạn sẽ kiểm thêm gì?
4. 1/5 số dòng còn lại không có `Customer ID`. Báo cáo doanh thu giữ chúng; báo cáo "doanh thu mỗi khách" thì sao?

Xong thì mở app, làm phần **trắc nghiệm** của bài.